

---


# <h1 align=center> **Sistema de recomendacion de peliculas** </h1>




---


# <h1 align=center> **Modelo de Machine Learning** </h1>




---



Se trabajará con los siguientes métodos y técnicas de Procesamiento de Lenguaje Natural:

TF-IDF (Frecuencia de Término - Frecuencia Inversa de Documento): Este enfoque transforma el texto en una representación numérica, creando una matriz de características donde cada palabra está asociada a un valor que indica su relevancia en el documento frente al conjunto de documentos (corpus). Estos valores se calculan únicamente a partir de la frecuencia de las palabras en el texto.

Similitud del Coseno: Esta técnica evalúa la similitud entre dos vectores generados a partir del TF-IDF, midiendo qué tan cercanos son en un espacio vectorial. La similitud se basa exclusivamente en los términos y sus ponderaciones en el texto, sin verse influenciada por otros valores numéricos externos.

##Ingesta de datos luego de aplicado el EDA

In [ ]:
#Importar librerías
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from wordcloud import WordCloud, STOPWORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Cargar el DataFrame parte 1
df = pd.read_parquet('/content/drive/My Drive/HENRY/ProyectoIndividual/data_preparada.parquet')


In [ ]:
df.dtypes

,0
budget,float64
id,int64
original_language,object
overview,object
popularity,float64
release_date,datetime64[ns]
revenue,float64
runtime,float64
status,object
tagline,object


In [ ]:
df['title'][:30]

,title
0,Toy Story
1,Jumanji
2,Grumpier Old Men
3,Waiting to Exhale
4,Father of the Bride Part II
5,Heat
6,Sabrina
7,Tom and Huck
8,Sudden Death
9,GoldenEye


In [ ]:
df.columns

Index(['budget', 'id', 'original_language', 'overview', 'popularity',
       'release_date', 'revenue', 'runtime', 'status', 'tagline', 'title',
       'vote_average', 'vote_count', 'collection', 'genre', 'company',
       'country', 'language', 'release_year', 'return', 'actor', 'director'],
      dtype='object')

In [ ]:
#revisar valores nulos
df.isnull().sum()

,0
budget,0
id,0
original_language,11
overview,941
popularity,0
release_date,0
revenue,0
runtime,246
status,80
tagline,24959


In [ ]:
df.shape

(45346, 22)

**Se crearán dos columnas como resultado del análisis efectuado en el EDA**

In [ ]:
# Extraer el primer actor de la lista en la columna 'actors'
df['first_actor'] = df['actor'].apply(lambda x: x.split(',')[0] if pd.notna(x) else '')

# Extraer el primer director de la lista en la columna 'director'
df['first_director'] = df['director'].apply(lambda x: x.split(',')[0] if pd.notna(x) else '')

In [ ]:
df.head()

,budget,id,original_language,overview,popularity,release_date,revenue,runtime,status,tagline,...,genre,company,country,language,release_year,return,actor,director,first_actor,first_director
0,30000000.0,862,en,"Led by Woody, Andy's toys live happily in his ...",21.946943,1995-10-30,373554033.0,81.0,Released,None,...,"Animation, Comedy, Family",Pixar Animation Studios,United States of America,English,1995,12.45,"Tom Hanks, Tim Allen, Don Rickles, Jim Varney,...",John Lasseter,Tom Hanks,John Lasseter
1,65000000.0,8844,en,When siblings Judy and Peter discover an encha...,17.015539,1995-12-15,262797249.0,104.0,Released,Roll the dice and unleash the excitement!,...,"Adventure, Fantasy, Family","TriStar Pictures, Teitler Film, Interscope Com...",United States of America,"English, Français",1995,4.04,"Robin Williams, Jonathan Hyde, Kirsten Dunst, ...",Joe Johnston,Robin Williams,Joe Johnston
2,0.0,15602,en,A family wedding reignites the ancient feud be...,11.712900,1995-12-22,0.0,101.0,Released,Still Yelling. Still Fighting. Still Ready for...,...,"Romance, Comedy","Warner Bros., Lancaster Gate",United States of America,English,1995,0.00,"Walter Matthau, Jack Lemmon, Ann-Margret, Soph...",Howard Deutch,Walter Matthau,Howard Deutch
3,16000000.0,31357,en,"Cheated on, mistreated and stepped on, the wom...",3.859495,1995-12-22,81452156.0,127.0,Released,Friends are the people who let you be yourself...,...,"Comedy, Drama, Romance",Twentieth Century Fox Film Corporation,United States of America,English,1995,5.09,"Whitney Houston, Angela Bassett, Loretta Devin...",Forest Whitaker,Whitney Houston,Forest Whitaker
4,0.0,11862,en,Just when George Banks has recovered from his ...,8.387519,1995-02-10,76578911.0,106.0,Released,Just When His World Is Back To Normal... He's ...,...,Comedy,"Sandollar Productions, Touchstone Pictures",United States of America,English,1995,0.00,"Steve Martin, Diane Keaton, Martin Short, Kimb...",Charles Shyer,Steve Martin,Charles Shyer


In [ ]:
df.columns

Index(['budget', 'id', 'original_language', 'overview', 'popularity',
       'release_date', 'revenue', 'runtime', 'status', 'tagline', 'title',
       'vote_average', 'vote_count', 'collection', 'genre', 'company',
       'country', 'language', 'release_year', 'return', 'actor', 'director',
       'first_actor', 'first_director'],
      dtype='object')

Se utilizará el atributo overview en combinación de otros más para evaluar los distintos modelos

In [ ]:
df = df.fillna('')  # Reemplazar nulos con cadenas vacias

##**Modelo**

In [ ]:
#Extrar las columnas relevantes para el modelo
model = df[['id','title', 'overview', 'genre', 'first_director']].copy()

In [ ]:
##Se separan los géneros y se convierten en palabras individuales
model['genre'] = model['genre'].fillna('').apply(lambda x: ' '.join(x.replace(',', ' ').replace('-', '').lower().split()))

In [ ]:
# Se crea una instancia de la clase TfidfVectorizer
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
# Aplicar la transformación TF-IDF al texto contenido en las columnas "overview_clean", "genres" y "director" del dataframe 'modelo4'
tfidf_matriz = tfidf.fit_transform(model['overview'] + ' ' + model['genre'] + ' ' + model['first_director'])

In [ ]:
tfidf_matriz.shape

(45346, 1107314)

In [ ]:
# Función para obtener recomendaciones
def recomendacion(titulo):
    #Crear una serie que asigna un índice a cada título de las películas
    indices = pd.Series(model.index, index=model['title']).drop_duplicates()
    if titulo not in indices:
        return 'La película ingresada no se encuentra en la base de datos'
    else:
        #Obtener el índice de la película que coincide con el título
        ind = pd.Series(indices[titulo]) if titulo in indices else None
        #Si el título de la película está duplicado, devolver el índice de la primera aparición del título en el DataFrame
        if model.duplicated(['title']).any():
            primer_ind = model[model['title'] == titulo].index[0]
            if not ind.equals(pd.Series(primer_ind)):
                ind = pd.Series(primer_ind)
        #Calcular la similitud coseno entre la película de entrada y todas las demás películas en la matriz de características
        cosine_sim = cosine_similarity(tfidf_matriz[ind], tfidf_matriz).flatten()
        simil = sorted(enumerate(cosine_sim), key=lambda x: x[1], reverse=True)[1:6]
        #Verificar que los índices obtenidos son válidos
        valid_ind = [i[0] for i in simil if i[0] < len(model)]
        #Obtener los títulos de las películas más similares utilizando el índice de cada película
        recomendaciones = model.iloc[valid_ind]['title'].tolist()
        #Devolver la lista de títulos de las películas recomendadas
        return recomendaciones

In [ ]:
recomendacion('Toy Story')

['Toy Story 2',
 'Toy Story 3',
 'Superstar Goofy',
 'Small Fry',
 'The 40 Year Old Virgin']

In [ ]:
recomendacion('Balto')

['Cheburashka',
 'Mars Needs Moms',
 'The Four Feathers',
 'Кентервильское привидение',
 'Investigation Held by Kolobki']

In [ ]:
recomendacion('Billy Madison')

['Skipped Parts',
 'Jean-Michel Basquiat: The Radiant Child',
 'Lonesome Jim',
 'A Hole in the Head',
 'Swimfan']

In [ ]:
recomendacion('Interstellar')

['Voices of a Distant Star',
 'Suburban Commando',
 'Stargate',
 'Following',
 'Holidays by the Sea']

In [ ]:
recomendacion('Ratatouille')

['Superstar Goofy',
 'Your Friend the Rat',
 'Babysitting',
 'Tatu and Patu',
 'Le Grand Restaurant']

Se crea un dataframe de las peliculas recomendadas para desplegar los resultados en Render.

In [ ]:
movies_df = model[['title']]

In [ ]:
# Generar un DataFrame con todas las titulos de las peliculas donde se visualice una columna con la lista de peliculas recomendas.
# OJO puede tardar bastante en ejecutarse pero esto permitira tener la informacion completa para deployar en RENDER.

# Lista para almacenar los resultados.
recommendation_results = []

# Iterar sobre cada título de película en la columna 'title' del DataFrame 'movies_df'.
for title in movies_df['title']:
    # Obtener las recomendaciones para cada título utilizando la función `recomendacion`.
    recomendaciones_lista = recomendacion(title)  # Esto debería devolver una lista de recomendaciones.

    # Verificar si la función devuelve una lista antes de unir.
    if isinstance(recomendaciones_lista, list):
        recomendacion_concatenada = ", ".join(recomendaciones_lista)
    else:

        recomendacion_concatenada = "No se encontraron recomendaciones"

    # Añadir el resultado al listado.
    recommendation_results.append({'title': title, 'recommendation': recomendacion_concatenada})

# Crear el DataFrame con los resultados.
all_recommendations_df = pd.DataFrame(recommendation_results)



In [ ]:
display(all_recommendations_df)

,title,recommendation
0,Toy Story,"Toy Story 2, Toy Story 3, Superstar Goofy, Sma..."
1,Jumanji,"Not Safe for Work, Table No. 21, Word Wars, Le..."
2,Grumpier Old Men,"Şevkat Yerimdar, No Man's Island, The Odd Coup..."
3,Waiting to Exhale,"Everything's Fine, We're Leaving, First Daught..."
4,Father of the Bride Part II,"Father of the Bride, Kuffs, Baby Boom, North t..."
...,...,...
45341,Robin Hood,"The Story of Robin Hood and His Merrie Men, Wi..."
45342,Century of Birthing,"Butterflies Have No Memories, Naked Under the ..."
45343,Betrayal,"Jacknife, Clash by Night, Written on the Wind,..."
45344,Satan Triumphant,"One Foot in Heaven, Ali G Indahouse, The Matri..."


In [ ]:
# Generar un archivo csv y un parquet de df all_recommendations_df y guardarlo en drive

import pandas as pd

# Guardar el DataFrame como un archivo CSV en Google Drive
all_recommendations_df.to_csv('/content/drive/My Drive/HENRY/ProyectoIndividual/all_recommendations_df.csv', index=False)

# Guardar el DataFrame como un archivo Parquet en Google Drive
all_recommendations_df.to_parquet('/content/drive/My Drive/HENRY/ProyectoIndividual/all_recommendations_df.parquet', index=False)


In [ ]:
# Cargar el DataFrame de las peliculas recomendadas (Se crea el df a partir del archivo de texto parquet, dado que si se vuelve a ejecutar el codigo que crea el df con toasa las recomendaciones puede tardar demasiado )
all_recommendations_df= pd.read_parquet('/content/drive/My Drive/HENRY/ProyectoIndividual/all_recommendations_df.parquet')
display(all_recommendations_df)


,title,recommendation
0,Toy Story,"Toy Story 2, Toy Story 3, Superstar Goofy, Sma..."
1,Jumanji,"Not Safe for Work, Table No. 21, Word Wars, Le..."
2,Grumpier Old Men,"Şevkat Yerimdar, No Man's Island, The Odd Coup..."
3,Waiting to Exhale,"Everything's Fine, We're Leaving, First Daught..."
4,Father of the Bride Part II,"Father of the Bride, Kuffs, Baby Boom, North t..."
...,...,...
45341,Robin Hood,"The Story of Robin Hood and His Merrie Men, Wi..."
45342,Century of Birthing,"Butterflies Have No Memories, Naked Under the ..."
45343,Betrayal,"Jacknife, Clash by Night, Written on the Wind,..."
45344,Satan Triumphant,"One Foot in Heaven, Ali G Indahouse, The Matri..."


In [ ]:
# Combinar  los dataframes df y all_recommendations_df eliminar duplicado el datframe se llama data_preparada_ML
# Combinar los DataFrames df y all_recommendations_df
data_preparada_ML = pd.merge(df, all_recommendations_df, on='title', how='left')
# Eliminar duplicados (si es necesario)
data_preparada_ML = data_preparada_ML.drop_duplicates()
# Mostrar el DataFrame resultante
display(data_preparada_ML)

,budget,id,original_language,overview,popularity,release_date,revenue,runtime,status,tagline,...,company,country,language,release_year,return,actor,director,first_actor,first_director,recommendation
0,30000000.0,862,en,"Led by Woody, Andy's toys live happily in his ...",21.946943,1995-10-30,373554033.0,81.0,Released,,...,Pixar Animation Studios,United States of America,English,1995,12.45,"Tom Hanks, Tim Allen, Don Rickles, Jim Varney,...",John Lasseter,Tom Hanks,John Lasseter,"Toy Story 2, Toy Story 3, Superstar Goofy, Sma..."
1,65000000.0,8844,en,When siblings Judy and Peter discover an encha...,17.015539,1995-12-15,262797249.0,104.0,Released,Roll the dice and unleash the excitement!,...,"TriStar Pictures, Teitler Film, Interscope Com...",United States of America,"English, Français",1995,4.04,"Robin Williams, Jonathan Hyde, Kirsten Dunst, ...",Joe Johnston,Robin Williams,Joe Johnston,"Not Safe for Work, Table No. 21, Word Wars, Le..."
2,0.0,15602,en,A family wedding reignites the ancient feud be...,11.712900,1995-12-22,0.0,101.0,Released,Still Yelling. Still Fighting. Still Ready for...,...,"Warner Bros., Lancaster Gate",United States of America,English,1995,0.00,"Walter Matthau, Jack Lemmon, Ann-Margret, Soph...",Howard Deutch,Walter Matthau,Howard Deutch,"Şevkat Yerimdar, No Man's Island, The Odd Coup..."
3,16000000.0,31357,en,"Cheated on, mistreated and stepped on, the wom...",3.859495,1995-12-22,81452156.0,127.0,Released,Friends are the people who let you be yourself...,...,Twentieth Century Fox Film Corporation,United States of America,English,1995,5.09,"Whitney Houston, Angela Bassett, Loretta Devin...",Forest Whitaker,Whitney Houston,Forest Whitaker,"Everything's Fine, We're Leaving, First Daught..."
4,0.0,11862,en,Just when George Banks has recovered from his ...,8.387519,1995-02-10,76578911.0,106.0,Released,Just When His World Is Back To Normal... He's ...,...,"Sandollar Productions, Touchstone Pictures",United States of America,English,1995,0.00,"Steve Martin, Diane Keaton, Martin Short, Kimb...",Charles Shyer,Steve Martin,Charles Shyer,"Father of the Bride, Kuffs, Baby Boom, North t..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54261,0.0,30840,en,"Yet another version of the classic epic, with ...",5.683753,1991-05-13,0.0,104.0,Released,,...,"Westdeutscher Rundfunk (WDR), Working Title Fi...","Canada, Germany, United Kingdom, United States...",English,1991,0.00,"Patrick Bergin, Uma Thurman, David Morrissey, ...",John Irvin,Patrick Bergin,John Irvin,"The Story of Robin Hood and His Merrie Men, Wi..."
54265,0.0,111109,tl,An artist struggles to finish his work while a...,0.178241,2011-11-17,0.0,360.0,Released,,...,Sine Olivia,Philippines,,2011,0.00,"Angel Aquino, Perry Dizon, Hazel Orencio, Joel...",Lav Diaz,Angel Aquino,Lav Diaz,"Butterflies Have No Memories, Naked Under the ..."
54266,0.0,67758,en,"When one of her hits goes wrong, a professiona...",0.903007,2003-08-01,0.0,90.0,Released,A deadly game of wits.,...,American World Pictures,United States of America,English,2003,0.00,"Erika Eleniak, Adam Baldwin, Julie du Page, Ja...",Mark L. Lester,Erika Eleniak,Mark L. Lester,"Jacknife, Clash by Night, Written on the Wind,..."
54268,0.0,227506,en,"In a small town live two brothers, one a minis...",0.003503,1917-10-21,0.0,87.0,Released,,...,Yermoliev,Russia,,1917,0.00,"Iwan Mosschuchin, Nathalie Lissenko, Pavel Pav...",Yakov Protazanov,Iwan Mosschuchin,Yakov Protazanov,"One Foot in Heaven, Ali G Indahouse, The Matri..."


In [ ]:
# Convertir data_preparada_ML en formato parquet

# Convertir la columna 'runtime' a numérico, reemplazando valores no numéricos con NaN
data_preparada_ML['runtime'] = pd.to_numeric(data_preparada_ML['runtime'], errors='coerce')

# Guardar el DataFrame data_preparada_ML como un archivo Parquet en Google Drive
data_preparada_ML.to_parquet('/content/drive/My Drive/HENRY/ProyectoIndividual/data_preparada_ML.parquet', index=False)